# Checkpoint A -- the Part-1 recommender

This checkpoint closes **Part 1**. Over Units 1-6 you built five retrieval paths and an honest
scoreboard; here you assemble them into one recommender, measure it the way a real system is
measured, and give it a single final grade on data it has never seen.

**How this checkpoint works -- freeze, then score.**

- **Questions 1-6 run entirely on the `val` split.** You search the catalog, fit the Part-1 paths,
  measure the deepened metrics (precision@k, NDCG@k, coverage, diversity, novelty), build and tune a
  calibrated blend, and read a leave-one-path-out ablation. Every design decision -- which paths,
  which weights, which pool -- is chosen and FROZEN here, on `val`.
- **Question 7 unseals the sealed `test` split for the first and only time.** You score the FROZEN
  recommender on `test` in a single aggregate pass and interpret the result. You must not go back to
  Questions 1-6 to change anything after looking at `test`: a holdout you tune against stops being a
  holdout.

Fill the `None` / `"TODO"` slots, and answer each short-answer prompt in the string it names. Build
the generated recsys data (`recsys/data/generated/`) before you begin. Everything is measured at
`K = 10` with the 60 cold readers excluded, exactly as the frozen scoreboard does it.

In [ ]:
import json
from collections import defaultdict

import bookrec
import pandas as pd
from bookrec import (
    PathRegistry,
    blend,
    catalog_coverage,
    hit_rate_at_k,
    intra_list_diversity,
    ndcg_at_k,
    novelty,
    precision_at_k,
    rank,
    run_blended_scoreboard,
    run_validation_scoreboard,
    search_catalog,
)

SEED = 0
K = 10   # every metric is reported at the top-10 cutoff

# The same generated substrate Units 1-5 ran on.
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)        # the 2000-book catalog: the coverage denominator
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])   # 60 readers no personalized path can serve

# The interaction log as plain row dicts (the row-mapping log every path duck-types).
rows = pd.read_csv(interactions_path).to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set; their `val` positives are the
# held-out relevant set. The `test` split stays SEALED until the final question.
seen_by = defaultdict(set)
val_by = defaultdict(set)
for row in rows:
    if row["label"] != 1:
        continue
    if row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])
    elif row["split"] == "val":
        val_by[row["reader_id"]].add(row["item_id"])

warm_readers = sorted(reader for reader in val_by if reader not in cold_readers)

## Question 1 -- Know the shelf: search the catalog

Every recommender draws from a fixed **catalog** -- the universe of items it can return -- and that
same catalog is the denominator when we later ask how much of the shelf a recommender reaches. Use
the Unit-1 `search_catalog` helper to explore it.

Find every book in the `fantasy` genre, and (separately) every book published in `2011`. Report how
many of each there are, and the title of one catalog book so you can see the schema. Then, in one or
two sentences, state what the catalog is in this system and why its size is the right denominator for
a coverage metric.

In [ ]:
fantasy_books = None   # TODO: search_catalog(catalog, genre="fantasy") -- a list of Book objects
n_fantasy = None       # TODO: len(fantasy_books)

books_2011 = None      # TODO: search_catalog(catalog, year=2011)
n_2011 = None          # TODO: len(books_2011)

sample_title = None    # TODO: catalog[catalog_ids[0]].title  -- one book's title
catalog_size = None    # TODO: len(catalog_ids)

catalog_role = "TODO"  # 1-2 sentences: the catalog is the fixed universe of items every path draws
                       # from; its size is the coverage denominator (how much of the shelf we reach)

## Question 2 -- Assemble the Part-1 paths

A recommender is built from retrieval paths, each a `RetrievalPath` with the same `fit` / `retrieve`
contract. Build and `fit` the four Part-1 signal paths on the interaction log -- **popularity** (Unit
2), **lexical** (Unit 3), **item-item CF** (Unit 4), and **matrix factorization** (Unit 5; the MF fit
is the only slow one, about 15 s) -- and register all four in a `PathRegistry` so later questions can
loop over them by name. Then, as a sanity check, produce one warm reader's top-10 from a single path
with the retrieve-then-rank pattern, dropping the books that reader has already seen.

In [ ]:
popularity = None   # TODO: bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = None      # TODO: bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
item_item = None    # TODO: bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = None           # TODO: bookrec.MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)

registry = None     # TODO: a PathRegistry()
# TODO: register popularity, lexical, item_item, and mf in `registry`.
registered_names = None   # TODO: [path.name for path in registry]

# Sanity check: one warm reader's top-10 from the item-item path (retrieve-then-rank).
reader_id = None    # TODO: warm_readers[0]
seen = None         # TODO: seen_by[reader_id]
sample_recs = None  # TODO: rank(item_item.retrieve(reader_id, {"seen": seen}, K), n=K, exclude=seen)
sample_rec_ids = None  # TODO: [candidate.item_id for candidate in sample_recs]

## Question 3 -- Ranking metrics: precision@k and NDCG@k (not RMSE)

Hit@k only asks whether *any* relevant book reached the top k. Two richer **ranking metrics** look
deeper: **precision@k** is the fraction of the top-k slots that are relevant, and **NDCG@k**
additionally rewards putting a relevant book *higher* in the list. The frozen scoreboard reports all
of these; score each registered path on `val` with `run_validation_scoreboard` and collect its
hit@10, precision@10, and NDCG@10.

Then answer, in two or three sentences: this is an implicit-feedback top-k problem (a reader either
engaged a book or did not, and only ever sees a ranked shortlist). Why is a ranking metric the right
lens here rather than rating-prediction RMSE, and what does NDCG@k reward that hit@k and precision@k
both ignore?

In [ ]:
ranking_rows = []
for path in registry:                  # TODO: reuse the registry from Question 2
    board = None  # TODO: run_validation_scoreboard(path, interactions_path,
                  #        catalog_ids=catalog_ids, k=K, cold_readers=cold_readers)
                  #        -- split defaults to "val"; do NOT pass split here
    ranking_rows.append({
        "path": None,          # TODO: path.name
        "hit@10": None,        # TODO: board.hit_rate_at_k
        "precision@10": None,  # TODO: board.precision_at_k
        "ndcg@10": None,       # TODO: board.ndcg_at_k
    })

ranking_metrics_explanation = "TODO"  # 2-3 sentences: there are no held-out ratings to square-error
                                      # against, and the reader sees a ranked shortlist, so a ranking
                                      # metric fits; NDCG rewards HIGHER placement, which hit@k and
                                      # precision@k both ignore

## Question 4 -- Beyond accuracy: coverage, diversity, novelty

Accuracy is not the whole story. **Catalog coverage** is the fraction of the catalog a path ever
recommends to anyone; **intra-list diversity** of a reader's top-k is `1 - mean pairwise similarity`;
**novelty** is the mean self-information `-log2 p_hat` of the recommended books (rarer books are more
novel). Reusing the paths fitted in Question 2, measure the catalog coverage of matrix factorization
and of item-item CF, and for one warm reader compute the intra-list diversity of the item-item top-10
(using the fitted item-item cosine as the book-book similarity) and the novelty of both the item-item
and the popularity top-10.

Then, in two or three sentences, interpret what you find: MF ranks best (Question 3) yet reaches only
a sliver of the catalog, item-item reaches far more, and popularity's picks are the least novel. Why
can two paths with similar hit@k still serve very different readers?

In [ ]:
def coverage_of(name, path):
    # A single-path blend is just that path; the blended scoreboard reports its catalog coverage.
    return run_blended_scoreboard(
        {name: path}, interactions_path, catalog_ids=catalog_ids,
        k=K, pool=30, weights={name: 1.0}, cold_readers=cold_readers,
    ).catalog_coverage

mf_coverage = None          # TODO: coverage_of("mf", mf)               -- narrow (about 0.07)
item_item_coverage = None   # TODO: coverage_of("item-item", item_item) -- broad  (about 0.49)

# The fitted item-item cosine, addressed by item id, is the book-book similarity for diversity.
sim_col = None    # TODO: {item_id: j for j, item_id in enumerate(item_item.artifact()["item_ids"])}
similarity = None  # TODO: item_item.similarity  (the 2000 x 2000 cosine matrix)

reader = warm_readers[0]
reader_seen = seen_by[reader]
item_item_ids = None   # TODO: [c.item_id for c in
                       #        rank(item_item.retrieve(reader, {"seen": reader_seen}, 30),
                       #             n=K, exclude=reader_seen)]
popularity_ids = None  # TODO: the same top-10, from the popularity path

diversity_item_item = None  # TODO: intra_list_diversity(item_item_ids,
                            #        lambda a, b: similarity[sim_col[a], sim_col[b]])
novelty_item_item = None    # TODO: novelty(item_item_ids, popularity.counts)
novelty_popularity = None   # TODO: novelty(popularity_ids, popularity.counts)  -- LOWER (common books)

beyond_accuracy_explanation = "TODO"  # 2-3 sentences: equal hit@k can hide very different coverage
                                      # and novelty; MF narrow vs item-item broad; popularity least
                                      # novel because it recommends the same head books to everyone

## Question 5 -- Build the blend on `val`, then FREEZE it

Now assemble the first blended recommender. `blend` calibrates each path's scores to a common [0, 1]
scale, takes the weighted union of their candidate pools, and `rank(exclude=seen)` keeps the top k.
Two choices shape it: the per-path **retrieval pool** (how many candidates each path contributes) and
the per-path **weights**. Choose them on `val` and measure the result with `run_blended_scoreboard`:
report the blend's hit@10 and catalog coverage, and compare against matrix factorization, the best
single path on hit@10.

The pool and weights below are the configuration measured on this data in Unit 6 (pool 30; popularity
down-weighted because, as Question 6 shows, it adds head coverage rather than accuracy). Once you have
measured the blend on `val`, **FREEZE** every decision into `FROZEN_PATHS`, `FROZEN_WEIGHTS`, and
`FROZEN_POOL`, and record the frozen `val` hit@10 -- Question 7 scores exactly this recommender on
`test` and changes nothing. In two or three sentences, say honestly whether the blend beats the best
single path here, and why "a blend always wins" would be the wrong lesson to draw.

In [ ]:
FROZEN_POOL = 30
FROZEN_WEIGHTS = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}
FROZEN_PATHS = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}

blend_val = None   # TODO: run_blended_scoreboard(FROZEN_PATHS, interactions_path,
                   #        catalog_ids=catalog_ids, k=K, pool=FROZEN_POOL,
                   #        weights=FROZEN_WEIGHTS, cold_readers=cold_readers)
blend_val_hit = None       # TODO: blend_val.hit_rate_at_k        (about 0.306)
blend_val_coverage = None  # TODO: blend_val.catalog_coverage     (about 0.333)

mf_val = None              # TODO: run_blended_scoreboard({"mf": mf}, interactions_path,
                           #        catalog_ids=catalog_ids, k=K, pool=FROZEN_POOL,
                           #        weights={"mf": 1.0}, cold_readers=cold_readers)
mf_val_hit = None               # TODO: mf_val.hit_rate_at_k      (about 0.276)
blend_beats_best_single = None  # TODO: bool - blend_val_hit >= mf_val_hit

blend_reading = "TODO"  # 2-3 HONEST sentences: the blend wins on hit@10 AND coverage HERE, at this
                        # pinned config -- not a universal law; the config was frozen on val, not
                        # tuned against the test score you are about to read

## Question 6 -- Does every path earn its place? The leave-one-path-out ablation

A blend can carry dead weight. Measure each path's marginal value with a **leave-one-path-out
ablation** on `val`: drop one path, re-blend the other three at the same pool and their same weights,
and record the signed change in hit@10. A path that genuinely contributes makes hit@10 *fall* when it
is removed.

You will find that dropping lexical, item-item, or MF each lowers hit@10, but dropping **popularity**
does not -- its marginal accuracy value is about zero. In three or four sentences, explain the
mechanism (min-max **calibration** maps every path's top candidate to a score of 1.0, so a
reader-independent popularity path ties its head books with every reader's true top pick), and say why
the blend keeps popularity anyway -- what does it contribute if not hit@10, and which readers (those
with little or no history, for whom the personalized paths return an empty list) can it still serve?

In [ ]:
ablation_rows = []
for dropped in FROZEN_PATHS:
    kept = {name: path for name, path in FROZEN_PATHS.items() if name != dropped}
    kept_weights = {name: FROZEN_WEIGHTS[name] for name in kept}
    board = None  # TODO: run_blended_scoreboard(kept, interactions_path, catalog_ids=catalog_ids,
                  #        k=K, pool=FROZEN_POOL, weights=kept_weights, cold_readers=cold_readers)
    ablation_rows.append({
        "dropped": dropped,
        "blend_hit@10": None,   # TODO: board.hit_rate_at_k
        "delta_vs_full": None,  # TODO: board.hit_rate_at_k - blend_val_hit  (signed)
    })

dropping_popularity_does_not_hurt = None  # TODO: bool - the delta_vs_full for "popularity" is >= 0.0
ablation_explanation = "TODO"  # 3-4 sentences: min-max calibration ties a flat, reader-independent
                               # path at the top, so popularity's marginal hit@10 is about 0; we keep
                               # it for head coverage and robustness -- it still serves readers whose
                               # sparse history leaves the personalized paths returning []

## Question 7 -- Unseal `test`: one honest final score

The `test` split has been sealed since Unit 1. Now -- and only now, with every decision frozen in
Question 5 -- score the FROZEN recommender on `test` exactly once. Pass `split="test"` to
`run_blended_scoreboard` with the frozen paths, weights, and pool, and the same cold-reader exclusion.
Report the test hit@10, the test catalog coverage, and the number of readers scored, and compare them
to the `val` numbers you froze in Question 5.

Then write three or four sentences of honest interpretation. How do the `val` and `test` scores
compare, and roughly how many held-out readers does each aggregate over? And the hygiene question at
the heart of this checkpoint: why do you evaluate `test` exactly once and never tune any decision
against it -- what would the final number stop meaning if you went back and adjusted the blend after
seeing it?

In [ ]:
# The authorized one-shot unseal of the sealed `test` holdout. Run this ONCE, and do not return to
# Questions 1-6 to change anything after reading it.
blend_test = None   # TODO: run_blended_scoreboard(FROZEN_PATHS, interactions_path,
                    #        catalog_ids=catalog_ids, k=K, pool=FROZEN_POOL,
                    #        weights=FROZEN_WEIGHTS, cold_readers=cold_readers, split="test")
test_hit = None        # TODO: blend_test.hit_rate_at_k
test_coverage = None   # TODO: blend_test.catalog_coverage
test_readers = None    # TODO: blend_test.readers

# Compare against the frozen val numbers recorded in Question 5.
val_vs_test_hit = None  # TODO: (blend_val_hit, test_hit)

test_interpretation = "TODO"  # 3-4 sentences: val and test should be close (each aggregates over
                              # several hundred held-out readers after cold exclusion); test is scored
                              # ONCE because tuning any decision against it would make it a second
                              # validation set, and the number would no longer estimate performance on
                              # genuinely unseen readers